# 🌤️ Atmospheric GAN & VAE - Model Training Notebook

This notebook trains GAN and VAE models for synthetic atmospheric data generation.

**Steps:**
1. Generate physics-based training data
2. Train the GAN (Generative Adversarial Network)
3. Train the VAE (Variational Autoencoder)
4. Evaluate model quality
5. Download trained model files

**Make sure you are using a GPU runtime:** Runtime → Change runtime type → T4 GPU

## 1. Setup & Install Dependencies

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
import pickle
import json
import os
import zipfile
from google.colab import files

## 2. Generate Physics-Based Training Data

We generate realistic atmospheric data using ISA (International Standard Atmosphere) physics models, then use it to train the GAN and VAE.

In [ ]:
class AtmosphericPhysicsModel:
    """Physics-based atmospheric data generator for training data"""

    @staticmethod
    def calculate_temperature(altitude_m, surface_temp=15.0):
        if altitude_m <= 11000:
            return surface_temp + (-6.5 * altitude_m / 1000)
        else:
            return surface_temp + (-6.5 * 11)

    @staticmethod
    def calculate_pressure(altitude_m, surface_pressure=1013.25):
        temp_kelvin = 288.15
        return surface_pressure * (1 - 0.0065 * altitude_m / temp_kelvin) ** 5.255

    @staticmethod
    def calculate_humidity(altitude_m, surface_humidity=70):
        if altitude_m <= 2000:
            humidity = surface_humidity * (1 - altitude_m / 10000)
        else:
            humidity = surface_humidity * 0.8 * np.exp(-altitude_m / 8000)
        return max(5, min(100, humidity))


def generate_training_data(n_profiles=500, points_per_profile=100):
    """Generate diverse atmospheric training data with realistic variation"""
    model = AtmosphericPhysicsModel()
    all_data = []

    for profile_idx in range(n_profiles):
        # Randomize surface conditions for diversity
        surface_temp = np.random.uniform(-10, 40)
        surface_pressure = np.random.uniform(990, 1040)
        surface_humidity = np.random.uniform(30, 95)
        max_altitude = np.random.uniform(15000, 30000)

        altitudes = np.linspace(0, max_altitude, points_per_profile)

        for alt in altitudes:
            temp = model.calculate_temperature(alt, surface_temp) + np.random.normal(0, 1.5)
            pressure = model.calculate_pressure(alt, surface_pressure) + np.random.normal(0, 3)
            humidity = model.calculate_humidity(alt, surface_humidity) + np.random.normal(0, 5)
            humidity = np.clip(humidity, 5, 100)

            # Wind model: increases with altitude, jet stream effect
            wind_speed = 5 + (alt / 1000) * 3 + np.random.normal(0, 3)
            if 8000 < alt < 12000:  # Jet stream zone
                wind_speed += np.random.uniform(10, 40)
            wind_speed = max(0, wind_speed)

            wind_direction = 270 + np.random.normal(0, 40)  # Prevailing westerlies
            wind_direction = wind_direction % 360

            dewpoint = temp - np.random.uniform(2, 15)

            all_data.append([
                alt, temp, pressure, humidity,
                wind_speed, wind_direction, dewpoint
            ])

    columns = ['altitude_m', 'temperature_c', 'pressure_hpa',
               'humidity_percent', 'wind_speed_mps', 'wind_direction_deg', 'dewpoint_c']

    df = pd.DataFrame(all_data, columns=columns)
    return df


# Generate training data
print('Generating training data (500 profiles × 100 points = 50,000 records)...')
training_df = generate_training_data(n_profiles=500, points_per_profile=100)
print(f'Training data shape: {training_df.shape}')
print(f'\nStatistics:')
training_df.describe()

In [ ]:
# Visualize training data
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Training Data Distribution', fontsize=16)

sample = training_df.sample(5000)

axes[0, 0].scatter(sample['altitude_m'], sample['temperature_c'], alpha=0.1, s=1)
axes[0, 0].set_xlabel('Altitude (m)')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].set_title('Temperature vs Altitude')

axes[0, 1].scatter(sample['altitude_m'], sample['pressure_hpa'], alpha=0.1, s=1)
axes[0, 1].set_xlabel('Altitude (m)')
axes[0, 1].set_ylabel('Pressure (hPa)')
axes[0, 1].set_title('Pressure vs Altitude')

axes[0, 2].scatter(sample['altitude_m'], sample['humidity_percent'], alpha=0.1, s=1)
axes[0, 2].set_xlabel('Altitude (m)')
axes[0, 2].set_ylabel('Humidity (%)')
axes[0, 2].set_title('Humidity vs Altitude')

axes[1, 0].scatter(sample['altitude_m'], sample['wind_speed_mps'], alpha=0.1, s=1)
axes[1, 0].set_xlabel('Altitude (m)')
axes[1, 0].set_ylabel('Wind Speed (m/s)')
axes[1, 0].set_title('Wind Speed vs Altitude')

axes[1, 1].hist(sample['wind_direction_deg'], bins=36, edgecolor='black')
axes[1, 1].set_xlabel('Wind Direction (°)')
axes[1, 1].set_title('Wind Direction Distribution')

axes[1, 2].scatter(sample['temperature_c'], sample['dewpoint_c'], alpha=0.1, s=1)
axes[1, 2].set_xlabel('Temperature (°C)')
axes[1, 2].set_ylabel('Dewpoint (°C)')
axes[1, 2].set_title('Dewpoint vs Temperature')

plt.tight_layout()
plt.show()

## 3. Train the GAN Model

The GAN learns to generate individual atmospheric data points that statistically match the training distribution.

In [ ]:
class AtmosphericGAN:
    """GAN for generating synthetic atmospheric data"""

    def __init__(self, input_dim=7, latent_dim=100, learning_rate=0.0002):
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate

        self.generator = self._build_generator()
        self.discriminator = self._build_discriminator()
        self.gan = self._build_gan()

        self.scaler = StandardScaler()
        self.history = {'g_loss': [], 'd_loss': [], 'd_acc': []}

    def _build_generator(self):
        model = models.Sequential([
            layers.Dense(256, input_dim=self.latent_dim),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(momentum=0.8),

            layers.Dense(512),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(momentum=0.8),

            layers.Dense(1024),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(momentum=0.8),

            layers.Dense(self.input_dim, activation='tanh')
        ])
        return model

    def _build_discriminator(self):
        model = models.Sequential([
            layers.Dense(512, input_dim=self.input_dim),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),

            layers.Dense(256),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),

            layers.Dense(128),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),

            layers.Dense(1, activation='sigmoid')
        ])
        model.compile(
            loss='binary_crossentropy',
            optimizer=optimizers.Adam(learning_rate=self.learning_rate, beta_1=0.5),
            metrics=['accuracy']
        )
        return model

    def _build_gan(self):
        self.discriminator.trainable = False
        model = models.Sequential([self.generator, self.discriminator])
        model.compile(
            loss='binary_crossentropy',
            optimizer=optimizers.Adam(learning_rate=self.learning_rate, beta_1=0.5)
        )
        return model

    def train(self, X_train, epochs=2000, batch_size=64, verbose=1):
        X_train_scaled = self.scaler.fit_transform(X_train)
        real_labels = np.ones((batch_size, 1))
        fake_labels = np.zeros((batch_size, 1))

        for epoch in range(epochs):
            # Train Discriminator
            idx = np.random.randint(0, X_train_scaled.shape[0], batch_size)
            real_data = X_train_scaled[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            fake_data = self.generator.predict(noise, verbose=0)

            d_loss_real = self.discriminator.train_on_batch(real_data, real_labels)
            d_loss_fake = self.discriminator.train_on_batch(fake_data, fake_labels)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train Generator
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.gan.train_on_batch(noise, real_labels)

            self.history['d_loss'].append(d_loss[0])
            self.history['d_acc'].append(d_loss[1])
            self.history['g_loss'].append(g_loss)

            if verbose and epoch % 200 == 0:
                print(f'Epoch {epoch}/{epochs} — D loss: {d_loss[0]:.4f}, '
                      f'D acc: {d_loss[1]:.4f}, G loss: {g_loss:.4f}')

        print('\n✅ GAN training complete!')

    def generate_samples(self, n_samples):
        noise = np.random.normal(0, 1, (n_samples, self.latent_dim))
        generated_scaled = self.generator.predict(noise, verbose=0)
        return self.scaler.inverse_transform(generated_scaled)


# Initialize and train GAN
print('🚀 Initializing GAN...')
gan = AtmosphericGAN(input_dim=7, latent_dim=100)

training_array = training_df.values

print('\n🏋️ Training GAN (2000 epochs)...')
print('This takes ~3-5 minutes on GPU\n')
gan.train(training_array, epochs=2000, batch_size=64)

In [ ]:
# Plot GAN training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(gan.history['d_loss'], alpha=0.7)
axes[0].set_title('Discriminator Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(gan.history['g_loss'], alpha=0.7, color='orange')
axes[1].set_title('Generator Loss')
axes[1].set_xlabel('Epoch')

axes[2].plot(gan.history['d_acc'], alpha=0.7, color='green')
axes[2].set_title('Discriminator Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Ideal (0.5)')
axes[2].legend()

plt.suptitle('GAN Training History', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Generate and compare GAN samples vs real data
gan_samples = gan.generate_samples(5000)
columns = training_df.columns.tolist()
gan_df = pd.DataFrame(gan_samples, columns=columns)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('GAN Generated (orange) vs Real Training Data (blue)', fontsize=14)

real_sample = training_df.sample(5000)

for idx, col in enumerate(['temperature_c', 'pressure_hpa', 'humidity_percent',
                           'wind_speed_mps', 'wind_direction_deg', 'dewpoint_c']):
    ax = axes[idx // 3, idx % 3]
    ax.hist(real_sample[col], bins=50, alpha=0.5, label='Real', density=True)
    ax.hist(gan_df[col], bins=50, alpha=0.5, label='GAN', density=True)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

print('\nStatistical Comparison:')
print('\n--- Real Data ---')
print(real_sample.describe().round(2))
print('\n--- GAN Generated ---')
print(gan_df.describe().round(2))

## 4. Train the VAE Model

The VAE generates time-series sequences of atmospheric data (e.g., a complete radiosonde profile).

In [ ]:
class TimeSeriesVAE:
    """Variational Autoencoder for time-series atmospheric data"""

    def __init__(self, sequence_length=50, n_features=7, latent_dim=20):
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.latent_dim = latent_dim
        self.scaler = StandardScaler()
        self.encoder, self.decoder, self.vae = self._build_vae()

    def _build_vae(self):
        # Encoder
        encoder_inputs = layers.Input(shape=(self.sequence_length, self.n_features))
        x = layers.LSTM(128, return_sequences=True)(encoder_inputs)
        x = layers.LSTM(64)(x)

        z_mean = layers.Dense(self.latent_dim, name='z_mean')(x)
        z_log_var = layers.Dense(self.latent_dim, name='z_log_var')(x)

        # Sampling
        def sampling(args):
            z_mean, z_log_var = args
            batch = tf.shape(z_mean)[0]
            dim = tf.shape(z_mean)[1]
            epsilon = tf.random.normal(shape=(batch, dim))
            return z_mean + tf.exp(0.5 * z_log_var) * epsilon

        z = layers.Lambda(sampling, name='z')([z_mean, z_log_var])

        encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')

        # Decoder
        latent_inputs = layers.Input(shape=(self.latent_dim,))
        x = layers.Dense(64)(latent_inputs)
        x = layers.RepeatVector(self.sequence_length)(x)
        x = layers.LSTM(64, return_sequences=True)(x)
        x = layers.LSTM(128, return_sequences=True)(x)
        decoder_outputs = layers.TimeDistributed(layers.Dense(self.n_features))(x)

        decoder = models.Model(latent_inputs, decoder_outputs, name='decoder')

        # VAE
        outputs = decoder(encoder(encoder_inputs)[2])
        vae = models.Model(encoder_inputs, outputs, name='vae')

        # VAE Loss
        reconstruction_loss = tf.reduce_mean(
            tf.keras.losses.mse(encoder_inputs, outputs)
        ) * self.sequence_length * self.n_features

        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
        )

        vae.add_loss(reconstruction_loss + kl_loss)
        vae.compile(optimizer='adam')

        return encoder, decoder, vae

    def prepare_sequences(self, data):
        """Reshape flat data into time-series sequences"""
        n_samples = len(data) // self.sequence_length
        data_trimmed = data[:n_samples * self.sequence_length]

        # Normalize
        data_flat_scaled = self.scaler.fit_transform(data_trimmed)
        sequences = data_flat_scaled.reshape(n_samples, self.sequence_length, self.n_features)
        return sequences

    def train(self, data, epochs=100, batch_size=32):
        sequences = self.prepare_sequences(data)
        print(f'Training on {sequences.shape[0]} sequences of length {self.sequence_length}')

        history = self.vae.fit(
            sequences, sequences,
            epochs=epochs,
            batch_size=batch_size,
            verbose=1
        )
        print('\n✅ VAE training complete!')
        return history

    def generate_sequences(self, n_sequences):
        latent_samples = np.random.normal(0, 1, (n_sequences, self.latent_dim))
        generated_scaled = self.decoder.predict(latent_samples, verbose=0)

        generated_flat = generated_scaled.reshape(-1, self.n_features)
        generated = self.scaler.inverse_transform(generated_flat)
        return generated.reshape(n_sequences, self.sequence_length, self.n_features)


# Initialize and train VAE
SEQUENCE_LENGTH = 50
N_FEATURES = 7
LATENT_DIM = 20

print('🚀 Initializing VAE...')
vae = TimeSeriesVAE(sequence_length=SEQUENCE_LENGTH, n_features=N_FEATURES, latent_dim=LATENT_DIM)

print('\n🏋️ Training VAE (100 epochs)...')
print('This takes ~5-10 minutes on GPU\n')
vae_history = vae.train(training_array, epochs=100, batch_size=32)

In [ ]:
# Plot VAE training loss
plt.figure(figsize=(10, 4))
plt.plot(vae_history.history['loss'])
plt.title('VAE Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Generate and plot VAE sequences
n_gen = 10
vae_sequences = vae.generate_sequences(n_gen)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'VAE Generated Atmospheric Profiles ({n_gen} sequences)', fontsize=14)

cols_to_plot = ['Temperature (°C)', 'Pressure (hPa)', 'Humidity (%)', 'Wind Speed (m/s)']
feat_indices = [1, 2, 3, 4]  # temperature, pressure, humidity, wind_speed

for plot_idx, (feat_idx, title) in enumerate(zip(feat_indices, cols_to_plot)):
    ax = axes[plot_idx // 2, plot_idx % 2]
    for seq_idx in range(n_gen):
        ax.plot(vae_sequences[seq_idx, :, feat_idx], alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('Step')
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Save Trained Models

Save all model files so they can be uploaded to your project's `backend/trained_models/` directory.

In [ ]:
# Create output directory
os.makedirs('trained_models', exist_ok=True)

# Save GAN
print('Saving GAN model...')
gan.generator.save('trained_models/atmospheric_gan_generator.h5')
with open('trained_models/atmospheric_gan_scaler.pkl', 'wb') as f:
    pickle.dump(gan.scaler, f)
with open('trained_models/atmospheric_gan_history.json', 'w') as f:
    json.dump(gan.history, f)

# Save VAE
print('Saving VAE model...')
vae.decoder.save('trained_models/atmospheric_vae_decoder.h5')
with open('trained_models/atmospheric_vae_scaler.pkl', 'wb') as f:
    pickle.dump(vae.scaler, f)
vae_config = {
    'sequence_length': SEQUENCE_LENGTH,
    'n_features': N_FEATURES,
    'latent_dim': LATENT_DIM
}
with open('trained_models/atmospheric_vae_config.json', 'w') as f:
    json.dump(vae_config, f)

# List saved files
print('\n✅ Saved model files:')
for f in sorted(os.listdir('trained_models')):
    size = os.path.getsize(f'trained_models/{f}')
    print(f'  {f:50s} {size/1024:.1f} KB')

In [ ]:
# Create ZIP of all model files for easy download
print('Creating ZIP archive...')
with zipfile.ZipFile('trained_models.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk('trained_models'):
        for file in files_list:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, 'trained_models')
            zipf.write(filepath, arcname)

zip_size = os.path.getsize('trained_models.zip')
print(f'\n✅ Created trained_models.zip ({zip_size/1024/1024:.1f} MB)')
print('\nDownloading...')
files.download('trained_models.zip')

## 6. How to Deploy the Trained Models

After downloading `trained_models.zip`:

1. **Extract** the ZIP file
2. **Copy** all files into `backend/trained_models/` in your project:
   ```
   backend/trained_models/
   ├── atmospheric_gan_generator.h5
   ├── atmospheric_gan_scaler.pkl
   ├── atmospheric_gan_history.json
   ├── atmospheric_vae_decoder.h5
   ├── atmospheric_vae_scaler.pkl
   └── atmospheric_vae_config.json
   ```
3. **Commit and push** to GitHub
4. Your backend API will automatically load the models on startup!

### API Endpoints:
- `GET /api/models/status` — Check if models are loaded
- `POST /api/generate/gan` — Generate data using GAN
- `POST /api/generate/vae` — Generate time-series data using VAE